# Python → Java: Model Performance Benchmark

Compare Groq and Gemini models on the same Python program. Each model generates optimized Java, the notebook checks the output, and then ranks correct implementations by speedup.

**Leaderboard columns:** rank, model, Java execution time, and speedup vs. Python.

## Step 1 — Install dependencies

In [11]:
!pip -q install openai pandas

## Step 2 — Load API keys

In [19]:
from openai import OpenAI
from google import genai
import os
import sys
import time
import statistics
import subprocess
from pathlib import Path
import pandas as pd
from IPython.display import display
from dotenv import load_dotenv

load_dotenv()


groq_api_key = os.getenv("GROQ_API_KEY")

## Step 3 — Configure models

In [20]:
groq = OpenAI(
    api_key=groq_api_key,
    base_url="https://api.groq.com/openai/v1"
)

MODEL_CONFIG = {
    "Groq — GPT-OSS 120B": {
        "client": groq,
        "model": "openai/gpt-oss-120b"
    },
    "Groq — Qwen": {
        "client": groq,
        "model": "qwen/qwen3.8-27b"
    }
}

for name, config in MODEL_CONFIG.items():
    print(f"{name}: {config['model']}")

Groq — GPT-OSS 120B: openai/gpt-oss-120b
Groq — Qwen: qwen/qwen3.8-27b


## Step 4 — Create the Python → Java translator

The prompt asks for the fastest practical Java implementation while preserving the program's behavior.

In [21]:
SYSTEM_PROMPT = """
You are an expert performance engineer translating Python to Java.
Produce the fastest practical Java implementation while preserving behavior,
edge cases, numeric semantics, and exact printed output.
Improve the algorithm when possible. Prefer primitive types, avoid unnecessary
allocations and repeated work, and keep hot loops simple for the JVM.
Do not skip computation, hardcode results, or change the answer to improve speed.
Return only a complete, compilable Main.java source file.
"""

def port(model_name, python_code):
    config = MODEL_CONFIG[model_name]
    response = config["client"].chat.completions.create(
        model=config["model"],
        temperature=0.1,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": (
                "Convert this Python program to the fastest correct Java program. "
                "Use public class Main and preserve printed output.\n\n"
                f"Python code:\n```python\n{python_code}\n```"
            )}
        ]
    )
    code = response.choices[0].message.content.strip()
    code = re.sub(r"^```(?:java)?\s*|\s*```$", "", code, flags=re.IGNORECASE)
    return code.strip()


## Step 5 — Choose the benchmark

In [22]:
example_python = r'''
def calculate(iterations, param1, param2):
    result = 1.0
    for i in range(1, iterations + 1):
        j = i * param1 - param2
        result -= 1 / j
        j = i * param1 + param2
        result += 1 / j
    return result

result = calculate(200000000, 4, 1) * 4
print(f"Result: {result:.12f}")
'''

In [23]:
import gradio as gr
from pathlib import Path
import re

def convert_for_ui(model_label, python_code):
    try:
        java_code = port(model_label, python_code)
        
        # Save Python benchmark and generated Java to disk for comparison
        workdir = Path("model_comparison")
        workdir.mkdir(exist_ok=True)
        (workdir / "benchmark.py").write_text(python_code, encoding="utf-8")

        model_dir = workdir / re.sub(r"[^A-Za-z0-9_-]", "_", model_label)
        model_dir.mkdir(exist_ok=True)
        java_file = model_dir / "Main.java"
        java_file.write_text(java_code, encoding="utf-8")
        
        status_msg = f"Saved Java code to: {java_file.as_posix()} (ready for comparison!)"
        return java_code, status_msg
    except Exception as exc:
        return f"Conversion error: {type(exc).__name__}: {exc}", f"Error: {exc}"

with gr.Blocks() as app:
    gr.Markdown("# Python → Java Converter & Benchmark Preparer")
    with gr.Row():
        python_input = gr.Textbox(label="Python code", lines=24, value=example_python)
        java_output = gr.Code(label="Generated Java", language="python", lines=24)
    with gr.Row():
        model_dropdown = gr.Dropdown(
            choices=list(MODEL_CONFIG.keys()),
            value=list(MODEL_CONFIG.keys())[0],
            label="Select model",
        )
        convert_button = gr.Button("Convert to Java & Save File", variant="primary")
    status_output = gr.Textbox(label="Save Status", value="Ready to generate", interactive=False)

    convert_button.click(
        fn=convert_for_ui,
        inputs=[model_dropdown, python_input],
        outputs=[java_output, status_output],
    )

app.launch()


* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


## Step 6 — Compare all models

This cell measures the Python baseline, generates and compiles Java for each model, checks output equality, and ranks correct implementations. Java compilation time is excluded from execution time; JVM process startup is included.

In [24]:
def compare_models(python_code=None, repetitions=5, warmups=3, reuse_saved_java=True):
    workdir = Path("model_comparison")
    workdir.mkdir(exist_ok=True)
    py_file = workdir / "benchmark.py"
    
    # If python_code is provided, write it; otherwise read previously saved benchmark or use example
    if python_code is not None:
        py_file.write_text(python_code, encoding="utf-8")
    elif py_file.exists():
        python_code = py_file.read_text(encoding="utf-8")
    else:
        python_code = example_python
        py_file.write_text(python_code, encoding="utf-8")

    def run(command, timeout=180):
        start = time.perf_counter()
        result = subprocess.run(
            command, capture_output=True, text=True, timeout=timeout
        )
        elapsed = time.perf_counter() - start
        if result.returncode != 0:
            raise RuntimeError(result.stderr.strip() or "Command failed")
        return elapsed, result.stdout.strip()

    def clean_java_output(output):
        return "\n".join(
            line for line in output.splitlines()
            if not re.match(r"^\[.*\]\[warning\]\[os,container\]", line)
        ).strip()

    python_runs = [run([sys.executable, str(py_file)]) for _ in range(repetitions)]
    python_time = statistics.median(t for t, _ in python_runs)
    python_output = python_runs[0][1]

    if any(output != python_output for _, output in python_runs):
        raise RuntimeError("Python output changed between runs.")

    results = []

    for name, config in MODEL_CONFIG.items():
        row = {"Model": name, "Java time (s)": None, "Speedup": None, "Status": ""}
        try:
            model_dir = workdir / re.sub(r"[^A-Za-z0-9_-]", "_", name)
            model_dir.mkdir(exist_ok=True)
            java_file = model_dir / "Main.java"

            # Check if Java file was already saved from Gradio UI or earlier run
            if reuse_saved_java and java_file.exists():
                print(f"[{name}] Using saved Java file from {java_file}...")
                java_code = java_file.read_text(encoding="utf-8")
            else:
                print(f"[{name}] Generating Java via API...")
                java_code = port(name, python_code)
                java_file.write_text(java_code, encoding="utf-8")

            compiled = subprocess.run(
                ["javac", "-d", str(model_dir), str(java_file)],
                capture_output=True, text=True, timeout=180
            )
            if compiled.returncode != 0:
                raise RuntimeError("Compilation failed: " + compiled.stderr[-300:])

            command = ["java", "-cp", str(model_dir), "Main"]
            _, output = run(command)
            if clean_java_output(output) != python_output:
                row["Status"] = "Output mismatch"
                results.append(row)
                continue

            for _ in range(warmups):
                run(command)

            java_runs = [run(command)[0] for _ in range(repetitions)]
            java_time = statistics.median(java_runs)
            row["Java time (s)"] = round(java_time, 6)
            row["Speedup"] = round(python_time / java_time, 2)
            row["Status"] = "Correct"

        except Exception as error:
            row["Status"] = str(error).splitlines()[0][:100]

        results.append(row)

    df = pd.DataFrame(results)
    valid = df[df["Status"] == "Correct"].sort_values(
        "Speedup", ascending=False
    ).reset_index(drop=True)
    valid.insert(0, "Rank", range(1, len(valid) + 1))

    print(f"\nPython execution time: {python_time:.6f} seconds")
    print("Speedup = Python time / Java time\n")

    if not valid.empty:
        display(valid[["Rank", "Model", "Java time (s)", "Speedup"]])
        winner = valid.iloc[0]
        print(f"Fastest correct model: {winner['Model']} ({winner['Speedup']}x vs Python)")
    else:
        print("No model produced Java output matching Python.")

    failed = df[df["Status"] != "Correct"]
    if not failed.empty:
        print("\nModels not ranked:")
        display(failed[["Model", "Status"]])

    df.to_csv(workdir / "model_comparison_results.csv", index=False)
    print("\nFull results saved to model_comparison/model_comparison_results.csv")
    return valid


## Step 7 — Run the benchmark

The leaderboard ranks only implementations that compile and produce the same output as Python.

In [25]:
ranking = compare_models(
    example_python,
    repetitions=5,
    warmups=3
)

[Groq — GPT-OSS 120B] Using saved Java file from model_comparison\Groq___GPT-OSS_120B\Main.java...
[Groq — Qwen] Using saved Java file from model_comparison\Groq___Qwen\Main.java...

Python execution time: 50.508291 seconds
Speedup = Python time / Java time



,Rank,Model,Java time (s),Speedup
0,1,Groq — Qwen,0.981000,51.49
1,2,Groq — GPT-OSS 120B,1.427452,35.38


Fastest correct model: Groq — Qwen (51.49x vs Python)

Full results saved to model_comparison/model_comparison_results.csv
